# 06 — Validation plot with Gaussian distance circles (best epoch)

**Layout:** `GT | pred | GT−pred | r≤32 zoom | r≤16 zoom` for u, v, speed.

- **Cols 1–3:** full 128×128 field, distance circles r = **16, 32, 64** px
- **Col 3:** per-row `GT − pred` → **0–1**
- **Cols 4–5:** zoomed circle-masked error (no full-field frame), **0–1** from full-field min/max per row
- **Output:** `{RUN_DIR}/viz/samples_gauss_circles/sample_00_idx{idx}_epoch{best}.png`

In [6]:
import importlib
import json
from pathlib import Path

import torch

from src.train.adv_no_train import AdvNOTrainConfig
from src.train.fix_window_train import SimulationDataset, build_model, load_training_arrays
import src.train.fix_window_viz as fix_window_viz

importlib.reload(fix_window_viz)

from src.train.fix_window_viz import (
    _uv_with_speed,
    forward_fix_window_sample,
    load_pred_cache,
    pred_cache_path,
    save_gaussian_circles_sample_figure,
)

In [7]:
# --- run to plot (change here) ---
RUN_NAME = "adv_no_uv_16_19_dt96_full_l1"
RUN_DIR = Path("Models/adv_no") / RUN_NAME

SAMPLE_INDEX = 0         # sample slot (usually 0)
VAL_DS_IDX = None        # None = read from viz/viz_indices.json

CIRCLE_RADII = (16.0, 32.0, 64.0)
INNER_CIRCLE_RADII = (32.0, 16.0)  # cols 4–5: zoomed circle error
FIG_SCALE = 2.6
FIG_DPI = 150

DEVICE = "cuda:7" if torch.cuda.is_available() else "cpu"

In [8]:
cfg_path = RUN_DIR / f"train_config_{RUN_NAME}.json"
cfg = AdvNOTrainConfig.from_dict(json.loads(cfg_path.read_text()))

hist_path = RUN_DIR / "history" / f"training_history_{RUN_NAME}.pt"
hist = torch.load(hist_path, map_location="cpu", weights_only=False)
best_epoch = int(hist.get("best_epoch", 0))
best_val_l1 = float(hist.get("best_val_loss", float("nan")))
epochs = [best_epoch]

viz_idx_path = RUN_DIR / "viz" / "viz_indices.json"
if VAL_DS_IDX is not None:
    ds_idx = int(VAL_DS_IDX)
elif viz_idx_path.exists():
    ds_idx = int(json.loads(viz_idx_path.read_text())["indices"][SAMPLE_INDEX])
else:
    ds_idx = 57

out_dir = RUN_DIR / "viz" / "samples_gauss_circles"
out_dir.mkdir(parents=True, exist_ok=True)

sigma = cfg.loss_gaussian_sigma if cfg.loss_spatial_mode == "gaussian_center" else 32.0
radii = tuple(cfg.viz_gaussian_circle_radii) if hasattr(cfg, "viz_gaussian_circle_radii") else CIRCLE_RADII
if hasattr(cfg, "viz_gaussian_inner_circle_radii"):
    inner_radii = tuple(cfg.viz_gaussian_inner_circle_radii)
elif hasattr(cfg, "viz_gaussian_inner_circle_radius"):
    inner_radii = (float(cfg.viz_gaussian_inner_circle_radius), 16.0)
else:
    inner_radii = INNER_CIRCLE_RADII

print(f"Run: {RUN_DIR}")
print(f"Best epoch: {best_epoch} | val uv L1: {best_val_l1:.6f}")
print(f"Val index: {ds_idx} | spatial mode: {cfg.loss_spatial_mode}" + (f" loss σ={sigma}" if sigma else ""))
print(f"Circles r={radii} | zoom inner r={inner_radii} | fig_scale={FIG_SCALE} | dpi={FIG_DPI}")
print(f"Output: {out_dir}")

Run: Models/adv_no/adv_no_uv_16_19_dt96_full_l1
Best epoch: 56 | val uv L1: 0.648661
Val index: 57 | spatial mode: full σ=32.0
Gaussian circles: r=σ, 2σ | fig_scale=2.0 | dpi=150
Output: Models/adv_no/adv_no_uv_16_19_dt96_full_l1/viz/samples_gauss_circles


In [9]:
data, mask, day, hour, crop_h, crop_w = load_training_arrays(cfg)
device = torch.device(DEVICE)

n = len(data)
n_train = int(n * (1 - cfg.val_split))
train_set = SimulationDataset(data[:n_train], mask[:n_train], day[:n_train], hour[:n_train], iD=cfg.iD)
val_set = SimulationDataset(
    data[n_train:], mask[n_train:], day[n_train:], hour[n_train:],
    mean=train_set.mean, std=train_set.std, iD=cfg.iD,
)

best_ckpt = RUN_DIR / "best_model.pth"
if not best_ckpt.exists():
    raise FileNotFoundError(f"Missing {best_ckpt}")

model = build_model(crop_h, crop_w, cfg)
model.load_state_dict(torch.load(best_ckpt, map_location="cpu", weights_only=True))
model = model.to(device)

epoch = best_epoch
pred_cache_dir = RUN_DIR / "viz" / "pred_cache"
cache_file = pred_cache_path(pred_cache_dir, epoch, ds_idx)
if cache_file.exists():
    pred, target, m = load_pred_cache(cache_file)
    source = "pred_cache"
else:
    pred, target, m = forward_fix_window_sample(model, val_set, ds_idx, device, iD=cfg.iD)
    source = "best_model.pth"

print(f"Loaded prediction for epoch {epoch} from {source}")

Loaded prediction for epoch 56 from best_model.pth


In [ ]:
pred_plot, plot_names = _uv_with_speed(pred)
target_plot, _ = _uv_with_speed(target)

out_path = out_dir / f"sample_{SAMPLE_INDEX:02d}_idx{ds_idx}_epoch{best_epoch:03d}.png"
title = (
    f"best epoch {best_epoch} | val idx {ds_idx} | "
    f"full field | src: {source}"
)
loss_text = f"val uv L1={best_val_l1:.4f} | error panel = GT−pred (signed)"

save_gaussian_circles_sample_figure(
    pred_plot,
    target_plot,
    m,
    out_path,
    gaussian_sigma=sigma,
    channel_names=plot_names,
    title=title,
    loss_text=loss_text,
    fig_scale=FIG_SCALE,
    dpi=FIG_DPI,
    circle_radii=radii,
    inner_circle_radii=inner_radii,
)
print(f"saved {out_path}")

saved Models/adv_no/adv_no_uv_16_19_dt96_full_l1/viz/samples_gauss_circles/sample_00_idx57_epoch056.png
